In [ ]:
import os
import laspy
import numpy as np
from glob import glob
from tqdm import tqdm


In [ ]:
INPUT_DIR = ""
OUTPUT_DIR = ""

INDEX_ATTRIBUTE_NAME = "global_idx"

In [ ]:
def add_point_indices(input_dir, output_dir):

    print(f"--- Starting the point indexing process ---")
    print(f"Input directory: {input_dir}")
    print(f"Output directory: {output_dir}")

    os.makedirs(output_dir, exist_ok=True)

    search_paths = [os.path.join(input_dir, '*.laz'), os.path.join(input_dir, '*.las')]
    all_files = []
    for path in search_paths:
        all_files.extend(glob(path))

    if not all_files:
        print(f"\nERROR: No .laz or .las files found in {input_dir}")
        return

    print(f"\nFound {len(all_files)} files to process.")

    for file_path in tqdm(all_files, desc="Indexing files"):
        base_name = os.path.basename(file_path)
        output_path = os.path.join(output_dir, base_name)

        try:
            las = laspy.read(file_path)

            num_points = len(las.points)
            point_indices = np.arange(num_points, dtype=np.int64)

            if hasattr(las, INDEX_ATTRIBUTE_NAME):
                print(f"\nWarning: File {base_name} already has a '{INDEX_ATTRIBUTE_NAME}' attribute. It will be overwritten.")
                las[INDEX_ATTRIBUTE_NAME] = point_indices
            else:

                las.add_extra_dim(laspy.ExtraBytesParams(
                    name=INDEX_ATTRIBUTE_NAME,
                    type=np.int64,
                    description="Unique point index within file" 
                ))
                las[INDEX_ATTRIBUTE_NAME] = point_indices

            las.write(output_path)

        except Exception as e:
            print(f"\nAn error occurred while processing file {base_name}: {e}")

    print("\n--- Indexing process finished ---")
    print(f"All indexed files have been saved to: {output_dir}")


In [ ]:
add_point_indices(INPUT_DIR, OUTPUT_DIR)